In [1]:
#Import các thư viện cần thiết cho phần kiểm tra trong notebook
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

In [2]:
#Tải dữ liệu Iris và tạo DataFrame để quan sát
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0:iris.target_names[0], 
                                  1:iris.target_names[1], 
                                  2:iris.target_names[2]})

display(df.head())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [3]:
#Mô hình đánh giá riêng cho phần bonus sidebar
#Mô hình này chỉ dùng để đo accuracy, không phải mô hình dự đoán của app.
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    random_state=42,
    stratify=iris.target,
)

evaluation_model = GaussianNB()
evaluation_model.fit(X_train, y_train)
y_pred = evaluation_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f'Kích thước tập Train: {X_train.shape}')
print(f'Kích thước tập Test: {X_test.shape}')
print(f'Độ chính xác trên tập Test: {test_accuracy:.2%}')

Kích thước tập Train: (120, 4)
Kích thước tập Test: (30, 4)
Độ chính xác trên tập Test: 96.67%


In [4]:
!curl -L -A "/content/" \
-o setosa.jpg \
"https://upload.wikimedia.org/wikipedia/commons/1/11/Iris_setosa_2.jpg?utm_source=vi.wikipedia.org&utm_campaign=imageinfo&utm_content=original"
!curl -L -A "/content/" \
-o versicolor.jpg \
"https://upload.wikimedia.org/wikipedia/commons/2/27/Blue_Flag%2C_Ottawa.jpg?utm_source=en.wikipedia.org&utm_campaign=imageinfo&utm_content=original"
!curl -L -A "/content/" \
-o virginica.jpg \
"https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg?utm_source=en.wikipedia.org&utm_campaign=imageinfo&utm_content=original"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  3 3545k    3  131k    0     0   220k      0  0:00:16 --:--:--  0:00:16  220k
100 3545k  100 3545k    0     0  3716k      0 --:--:-- --:--:-- --:--:-- 3727k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 16  260k   16 43303    0     0   118k      0  0:00:02 --:--:--  0:00:02  120k
100  260k  100  260k    0     0   497k      0 --:--:-- --:--:-- --:--:--  501k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:

In [6]:
%%writefile app.py
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB


# Cấu hình trang phải
st.set_page_config(
    page_title="Demo Phân Loại Hoa Iris Dataset",
    layout="wide",
)

# Tải lại dữ liệu và tạo bảng phục vụ phần xem dữ liệu bonus.
def load_iris_dataframe():
    
    iris_data = load_iris()
    dataframe = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
    dataframe["target"] = iris_data.target
    dataframe["species"] = dataframe["target"].map(
        dict({0:iris_data.target_names[0], 
            1:iris_data.target_names[1], 
            2:iris_data.target_names[2]})
    )
    return iris_data, dataframe

# Mô hình chính được huấn luyện trên toàn bộ dữ liệu Iris và chỉ tạo một lần.
@st.cache_resource
def train_app_model():
    iris_data = load_iris()
    app_model = GaussianNB()
    app_model.fit(iris_data.data, iris_data.target)
    return app_model

# Mô hình đánh giá riêng dùng đúng phép chia 80/20 của phần bonus.
@st.cache_data
def calculate_test_accuracy():
    iris_data = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        iris_data.data,
        iris_data.target,
        test_size=0.2,
        random_state=42,
        stratify=iris_data.target,
    )
    evaluation_model = GaussianNB()
    evaluation_model.fit(X_train, y_train)
    predictions = evaluation_model.predict(X_test)
    return accuracy_score(y_test, predictions)

# Dự đoán tên loài và xác suất của cả ba lớp.
def predict_species(model, feature_values, target_names):
    input_data = np.array([feature_values], dtype=float)
    predicted_class = int(model.predict(input_data)[0])
    probabilities = model.predict_proba(input_data)[0]
    predicted_species = str(target_names[predicted_class])
    return predicted_species, probabilities

# Ánh xạ mỗi loài tới đúng file ảnh trong thư mục images.
def get_image_path(species_name):
    image_paths = {
        "setosa": Path("setosa.jpg"),
        "versicolor": Path("versicolor.jpg"),
        "virginica": Path("virginica.jpg"),
    }
    return Path(__file__).resolve().parent / image_paths[species_name]


iris, iris_df = load_iris_dataframe()
model = train_app_model()
test_accuracy = calculate_test_accuracy()

st.title("Demo Phân Loại Hoa Iris Dataset")
st.subheader("Mô hình: Gaussian Naive Bayes")
st.write(
    "Điều chỉnh bốn thông số ở cột bên trái, sau đó nhấn "
    "**Dự đoán ngay** để xem kết quả."
)

with st.sidebar:
    st.header("Thông tin mô hình")
    st.write("**Thuật toán:** Gaussian Naive Bayes")
    st.write("**Dữ liệu:** Iris Dataset")
    st.write("**Đánh giá:** Train/Test = 80/20")
    st.write("**random_state:** 42")
    st.write("**stratify:** y")
    st.metric("Độ chính xác trên tập Test", f"{test_accuracy:.2%}")
    st.caption(
        "Accuracy do mô hình đánh giá riêng tính trên tập Test; "
        "mô hình dự đoán của ứng dụng được huấn luyện trên toàn bộ dữ liệu."
    )

input_col, result_col = st.columns(2, gap="large")

with input_col:
    st.header("Thông số đầu vào")
    sepal_length = st.slider(
        "Chiều dài lá đài (Sepal Length - cm)",
        min_value=4.0, max_value=8.0, value=5.1, step=0.1,
    )
    sepal_width = st.slider(
        "Chiều rộng lá đài (Sepal Width - cm)",
        min_value=2.0, max_value=4.5, value=3.5, step=0.1,
    )
    petal_length = st.slider(
        "Chiều dài cánh hoa (Petal Length - cm)",
        min_value=1.0, max_value=7.0, value=1.4, step=0.1,
    )
    petal_width = st.slider(
        "Chiều rộng cánh hoa (Petal Width - cm)",
        min_value=0.1, max_value=2.5, value=0.2, step=0.1,
    )
    predict_button = st.button("Dự đoán ngay", type="primary", use_container_width=True)

with result_col:
    st.header("Kết quả dự đoán")
    if predict_button:
        feature_values = [sepal_length, sepal_width, petal_length, petal_width]
        species, probabilities = predict_species(
            model, feature_values, iris.target_names
        )

        st.markdown(f"## **{species.upper()}**")

        image_path = get_image_path(species)
        if image_path.is_file():
            st.image(str(image_path), caption=f"Hoa Iris {species.capitalize()}")
        else:
            st.warning(
                f"Không tìm thấy ảnh minh họa: {image_path}. "
                "Vui lòng thêm ảnh đúng tên vào thư mục images/."
            )

        st.subheader("Xác suất của từng loài")
        probability_df = pd.DataFrame(
            {"Loài": [name.capitalize() for name in iris.target_names],
             "Xác suất": probabilities}
        ).set_index("Loài")
        st.bar_chart(probability_df, y="Xác suất")
        for species_name, probability in probability_df["Xác suất"].items():
            st.write(f"**{species_name}:** {probability:.2%}")
    else:
        st.info("Nhấn **Dự đoán ngay** để hiển thị kết quả.")

with st.expander("Xem bảng dữ liệu Iris"):
    st.dataframe(iris_df, use_container_width=True)


Overwriting app.py


In [7]:
import subprocess
import sys
import time

from IPython.display import IFrame, display


# Khởi động Streamlit ở chế độ nền
streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.headless=true",
        "--server.port=8501",
        "--browser.gatherUsageStats=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Chờ server khởi động
time.sleep(3)

# Nhúng giao diện ngay trong notebook
display(IFrame(
    src="http://localhost:8501",
    width="100%",
    height=900,
))